In [28]:
import ast
import boto3
from dotenv import load_dotenv
load_dotenv()
import os
import pandas as pd
import sys

from FantAIno.utils.data_utils import get_secret
from FantAIno.constants import S3_GENERAL_PURPOSE_BUCKET_NAME

def hello_s3():
    """
    Use the AWS SDK for Python (Boto3) to create an Amazon Simple Storage Service
    (Amazon S3) client and list the buckets in your account.
    This example uses the default settings specified in your shared credentials
    and config files.
    """

    # Create an S3 client.
    s3_client = boto3.client(
        "s3",
        aws_access_key_id=get_secret("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=get_secret("AWS_SECRET_ACCESS_KEY"),
        # aws_session_token=get_secret("AWS_SESSION_TOKEN")
    )

    print("Hello, Amazon S3! Let's list your buckets:")

    # Create a paginator for the list_buckets operation.
    paginator = s3_client.get_paginator("list_buckets")

    # Use the paginator to get a list of all buckets.
    response_iterator = paginator.paginate(
        PaginationConfig={
            "PageSize": 50,  # Adjust PageSize as needed.
            "StartingToken": None,
        }
    )

    # Iterate through the pages of the response.
    buckets_found = False
    for page in response_iterator:
        if "Buckets" in page and page["Buckets"]:
            buckets_found = True
            for bucket in page["Buckets"]:
                print(f"\t{bucket['Name']}")

    if not buckets_found:
        print("No buckets found!")

if __name__ == "__main__":
    hello_s3()

Hello, Amazon S3! Let's list your buckets:
	fantaino-bucket-085777795487-us-east-2-an
	portfolio-assets-rk


In [5]:
# Create an S3 client.
s3_client = boto3.client("s3")

In [ ]:
import json

response = s3_client.get_object(Bucket=S3_GENERAL_PURPOSE_BUCKET_NAME, Key="3:33___In the Middle of Infinity.json")
json_content = json.load(response['Body'])
json_content

{'artist': '3:33',
 'album': 'In the Middle of Infinity',
 'genre': '[]',
 'rating': 6}

In [ ]:
response = s3_client.get_object(Bucket=S3_GENERAL_PURPOSE_BUCKET_NAME, Key="lyrics\$NOT___Ethereal.jsonl")
json_content = json.load(response['Body'])
json_content

{'artist': '$NOT',
 'album': 'Ethereal',
 'tracks': {'My World (Intro)': "[Intro]\nAyy, you know sometimes I be, I be shaking\nSometimes it's, it's with my demons and shit, you know?\nYeah\n\n[Chorus]\nAyy, I shake, I shackle (Why?)\nI battle all my demons (Why?)\nIn the mornin' I be fiendin' (Damn)\nIn the sleepin' I be dreamin' (Why?)\nWhat you know about this sadness? (What?)\nEven though I'm in a different planet (Why?)\nThere's Mars and Earth and Venus (Yeah)\nEverywhere I go, it's a magnet (Ah)\n\n[Verse 1]\nDown bad, I'ma tilt the axis (Why?)\nDo you wanna believe in magic? (Do you?)\nI don't think so, you might vanish (Uh-huh)\nMy bad, I don't want it to happen (Why?)\nNiggas want my clout and status (Why?)\nGot a Glock with a knife and a hatchet (Hatchet)\nSmoke a blunt with the leaf in thе packet (Packet)\nGot a blade and a Monclеr jacket (Jacket)\nWhen I die can you paint my casket? (Woah)\nWith the black paint in the basket (Damn)\nSay my name out loud in traffic (Traffic)\

In [7]:
from io import BytesIO
from PIL import Image

response = s3_client.get_object(Bucket=S3_GENERAL_PURPOSE_BUCKET_NAME, Key="album_art\$NOT___Ethereal.jpg")
image_content = Image.open(BytesIO(response['Body'].read()))
image_content.show()

In [ ]:
# get all lyrics
lyrics = []
paginator = s3_client.get_paginator('list_objects_v2')

for page in paginator.paginate(Bucket=S3_GENERAL_PURPOSE_BUCKET_NAME):
    for obj in page.get('Contents', []):
        if "lyrics\\" in obj['Key']:
            lyrics_obj = ast.literal_eval(s3_client.get_object(Bucket=S3_GENERAL_PURPOSE_BUCKET_NAME, Key=obj['Key'])['Body'].read().decode('utf-8'))
            lyrics.append(lyrics_obj)

In [18]:
len(lyrics)

3122

In [19]:
lyrics[0]

'{"artist": "$NOT", "album": "Ethereal", "tracks": {"My World (Intro)": "[Intro]\\nAyy, you know sometimes I be, I be shaking\\nSometimes it\'s, it\'s with my demons and shit, you know?\\nYeah\\n\\n[Chorus]\\nAyy, I shake, I shackle (Why?)\\nI battle all my demons (Why?)\\nIn the mornin\' I be fiendin\' (Damn)\\nIn the sleepin\' I be dreamin\' (Why?)\\nWhat you know about this sadness? (What?)\\nEven though I\'m in a different planet (Why?)\\nThere\'s Mars and Earth and Venus (Yeah)\\nEverywhere I go, it\'s a magnet (Ah)\\n\\n[Verse 1]\\nDown bad, I\'ma tilt the axis (Why?)\\nDo you wanna believe in magic? (Do you?)\\nI don\'t think so, you might vanish (Uh-huh)\\nMy bad, I don\'t want it to happen (Why?)\\nNiggas want my clout and status (Why?)\\nGot a Glock with a knife and a hatchet (Hatchet)\\nSmoke a blunt with the leaf in th\\u0435 packet (Packet)\\nGot a blade and a Moncl\\u0435r jacket (Jacket)\\nWhen I die can you paint my casket? (Woah)\\nWith the black paint in the basket (D

In [30]:
lyrics_df = pd.DataFrame(lyrics)
lyrics_df.head()

,artist,album,tracks
0,$NOT,Ethereal,"{'My World (Intro)': '[Intro] Ayy, you know so..."
1,$uicideboy$,I Want to Die in New Orleans,{'King Tulip': '[Intro: Max Beck] They changed...
2,$uicideboy$,Long Term Effects of SUFFERING,{'Degeneration in the Key of A Minor': '[Intro...
3,$uicideboy$,New World Depression,{'Lone Wolf Hysteria': '[Intro] G-R-E-Y ( You ...
4,070 Shake,You Cant Kill Me,{'Web': '[Verse] (What is your favorite) One t...
